# Phase 1 Failure Analysis — Interactive Notebook

Loads `per_image_metrics.csv` and provides interactive Plotly visualisations.
Run `infer_and_eval.py` first to generate the CSV.

**Class ordering (verified PR #11):** obstacle=0, water=1, sky=2  
**Models:** STDC2 (embedded), SegNeXt-T (embedded candidate), SegFormer MiT-B2 (upper bound)  
**Dataset:** LaRS val set, 198 images, 77 sequences

In [ ]:
import logging
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Set up file logger so progress is captured outside notebook
LOG_PATH = Path("logs/failure_analysis_notebook.log")
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler(LOG_PATH, mode="w"),
    ],
)
log = logging.getLogger(__name__)
log.info("Notebook started")

CSV_PATH = Path("per_image_metrics.csv")
df = pd.read_csv(CSV_PATH)
log.info(f"Loaded {len(df)} rows from {CSV_PATH}")

MODELS       = ["stdc2", "segnext_t", "segformer"]
MODEL_LABELS = {"stdc2": "STDC2", "segnext_t": "SegNeXt-T", "segformer": "SegFormer MiT-B2"}
CLASSES      = ["obstacle", "water", "sky"]
CLASS_COLORS = ["#e53935", "#1e88e5", "#00bcd4"]
FAIL_THRESH  = 0.5

print(f"DataFrame shape: {df.shape}")
df.head(3)

## 1. Per-image metric table
Sortable by any column. Filter to see worst images by obstacle IoU.

In [ ]:
# Worst 30 images by STDC2 obstacle IoU
worst30 = df.nsmallest(30, "stdc2_obstacle_iou").reset_index(drop=True)
display_cols = [
    "image_name", "sequence_id",
    "stdc2_obstacle_iou", "segnext_t_obstacle_iou", "segformer_obstacle_iou",
    "stdc2_miou", "segnext_t_miou", "segformer_miou",
]
worst30[display_cols].round(3)

## 2. IoU distribution — interactive box + strip plots

In [ ]:
# Melt to long format for plotly express
rows = []
for model in MODELS:
    for cls in CLASSES:
        col = f"{model}_{cls}_iou"
        for _, r in df.iterrows():
            rows.append({
                "model": MODEL_LABELS[model],
                "class": cls,
                "iou": r[col] * 100,
                "image": r["image_name"],
                "sequence": r["sequence_id"],
            })
long_df = pd.DataFrame(rows)

fig = px.box(
    long_df, x="model", y="iou", color="class",
    facet_col="class", points="outliers",
    color_discrete_map={"obstacle": CLASS_COLORS[0],
                        "water":    CLASS_COLORS[1],
                        "sky":      CLASS_COLORS[2]},
    title="Per-image IoU distributions — Phase 1 models × classes",
    labels={"iou": "IoU (%)", "model": ""},
    hover_data=["image", "sequence"],
)
fig.update_layout(height=500, showlegend=False)
fig.show()

## 3. Scatter: obstacle IoU — model comparison

In [ ]:
# STDC2 vs SegFormer obstacle IoU — coloured by SegNeXt-T
fig = px.scatter(
    df,
    x="stdc2_obstacle_iou", y="segformer_obstacle_iou",
    color="segnext_t_obstacle_iou",
    hover_data=["image_name", "sequence_id",
                "segnext_t_obstacle_iou", "stdc2_miou", "segformer_miou"],
    color_continuous_scale="RdYlGn",
    title="Obstacle IoU: STDC2 vs SegFormer (colour = SegNeXt-T)",
    labels={
        "stdc2_obstacle_iou": "STDC2 obstacle IoU",
        "segformer_obstacle_iou": "SegFormer obstacle IoU",
        "segnext_t_obstacle_iou": "SegNeXt-T IoU",
    },
)
fig.add_hline(y=FAIL_THRESH, line_dash="dash", line_color="red", opacity=0.5)
fig.add_vline(x=FAIL_THRESH, line_dash="dash", line_color="red", opacity=0.5)
fig.add_annotation(x=0.05, y=0.95, text=f"Failure threshold = {FAIL_THRESH}",
                   showarrow=False, font_color="red", xref="paper", yref="paper")
fig.show()

## 4. Cross-model failure overlap

In [ ]:
fail = {m: set(df[df[f"{m}_obstacle_iou"] < FAIL_THRESH]["image_name"]) for m in MODELS}
print(f"Threshold = {FAIL_THRESH}")
for m in MODELS:
    print(f"  {MODEL_LABELS[m]:>18s}: {len(fail[m])} failures")

all3    = fail["stdc2"] & fail["segnext_t"] & fail["segformer"]
emb_not = (fail["stdc2"] | fail["segnext_t"]) - fail["segformer"]
print(f"\nAll 3 models fail:                  {len(all3)} images")
print(f"Embedded only (not SegFormer fail):  {len(emb_not)} images (capacity gap)")
print(f"\nAll-3-failure images:")
for img in sorted(all3):
    row = df[df.image_name == img].iloc[0]
    print(f"  {img}: STDC2={row.stdc2_obstacle_iou:.3f}  SegNeXt={row.segnext_t_obstacle_iou:.3f}  "
          f"SegFormer={row.segformer_obstacle_iou:.3f}")

## 5. Sequence-level breakdown

In [ ]:
seq_records = []
for seq in sorted(df["sequence_id"].unique()):
    sub = df[df["sequence_id"] == seq]
    row = {"sequence_id": seq, "n_images": len(sub)}
    for m in MODELS:
        row[f"{m}_obs_iou"] = sub[f"{m}_obstacle_iou"].mean()
        row[f"{m}_miou"]    = sub[f"{m}_miou"].mean()
    seq_records.append(row)
seq_df = pd.DataFrame(seq_records).sort_values("stdc2_obs_iou")

# Melt for plotly
seq_long = pd.melt(
    seq_df,
    id_vars=["sequence_id", "n_images"],
    value_vars=[f"{m}_obs_iou" for m in MODELS],
    var_name="model", value_name="mean_obs_iou",
)
seq_long["model"] = seq_long["model"].map({
    f"{m}_obs_iou": MODEL_LABELS[m] for m in MODELS
})

fig = px.bar(
    seq_long, x="sequence_id", y="mean_obs_iou", color="model",
    barmode="group",
    title="Mean obstacle IoU per sequence (sorted by STDC2)",
    labels={"mean_obs_iou": "Mean obstacle IoU", "sequence_id": "Sequence"},
    color_discrete_map={v: c for v, c in zip(MODEL_LABELS.values(), ["#e53935", "#1e88e5", "#4caf50"])},
    hover_data=["n_images"],
)
fig.add_hline(y=FAIL_THRESH, line_dash="dash", line_color="red", opacity=0.6)
fig.update_xaxes(tickangle=60, tickfont_size=8)
fig.update_layout(height=500, xaxis_categoryorder="array",
                  xaxis_categoryarray=list(seq_df["sequence_id"]))
fig.show()

## 6. Per-class Dice / F1 distributions

In [ ]:
rows = []
for model in MODELS:
    for cls in CLASSES:
        col = f"{model}_{cls}_dice"
        for _, r in df.iterrows():
            rows.append({
                "model": MODEL_LABELS[model],
                "class": cls,
                "dice": r[col] * 100,
                "image": r["image_name"],
            })
dice_df = pd.DataFrame(rows)

fig = px.violin(
    dice_df, x="model", y="dice", color="class", box=True, points="outliers",
    facet_col="class",
    color_discrete_map={"obstacle": CLASS_COLORS[0],
                        "water":    CLASS_COLORS[1],
                        "sky":      CLASS_COLORS[2]},
    title="Per-image Dice/F1 distributions — Phase 1 models × classes",
    labels={"dice": "Dice/F1 (%)", "model": ""},
    hover_data=["image"],
)
fig.update_layout(height=500, showlegend=False)
fig.show()

## 7. Summary statistics

In [ ]:
summary = []
for m in MODELS:
    for cls in CLASSES:
        for metric in ["iou", "dice", "acc"]:
            col = f"{m}_{cls}_{metric}"
            vals = df[col].dropna() * 100
            summary.append({
                "model": MODEL_LABELS[m],
                "class": cls,
                "metric": metric.upper(),
                "mean": vals.mean(),
                "median": vals.median(),
                "p10": vals.quantile(0.10),
                "min": vals.min(),
            })
summary_df = pd.DataFrame(summary)
pd.set_option("display.float_format", "{:.2f}".format)
summary_df

In [ ]:
log.info("Notebook complete")